In [4]:
import torch

print(torch.__version__)

import inspect
import torch._library.infer_schema

print(inspect.getsource(torch._library.infer_schema))

2.5.0+cu124
# mypy: allow-untyped-decorators
# mypy: allow-untyped-defs
import inspect
import typing
from typing import List, Optional, Sequence, Union  # noqa: F401

import torch
from torch import device, dtype, Tensor, types
from torch.utils._exposed_in import exposed_in


@exposed_in("torch.library")
def infer_schema(
    prototype_function: typing.Callable,
    /,
    *,
    mutates_args,
    op_name: Optional[str] = None,
) -> str:
    r"""Parses the schema of a given function with type hints. The schema is inferred from the
    function's type hints, and can be used to define a new operator.

    We make the following assumptions:

    * None of the outputs alias any of the inputs or each other.
    * | String type annotations "device, dtype, Tensor, types" without library specification are
      | assumed to be torch.*. Similarly, string type annotations "Optional, List, Sequence, Union"
      | without library specification are assumed to be typing.*.
    * | Only the args list

In [5]:
from transformers import AutoProcessor, AutoModelForZeroShotImageClassification

processor = AutoProcessor.from_pretrained("openai/clip-vit-large-patch14")
model = AutoModelForZeroShotImageClassification.from_pretrained(
    "openai/clip-vit-large-patch14"
)

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [6]:
from PIL import Image

from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

In [8]:
prompt_raw = "the {object} is {state}"

entity_state_map = {"drawer": ["open", "closed"],
        }
images = []
solutions = []
for entity, states in entity_state_map.items():
    for state in states:
        for i in range(10):
            dir = f"../data/samples/front"
            image = Image.open(f"{dir}/{entity}_{state}_{i}.png")
            images.append(image)
            solutions.append(state)

p1 = prompt_raw.format(object="drawer", state="open")
p2 = prompt_raw.format(object="drawer", state="closed")
prompt = [p1, p2]

In [ ]:
inputs = processor(
    text=prompt,
    images=images,
    return_tensors="pt",  # type: ignore
    padding=True,  # type: ignore
)

outputs = model(**inputs)
logits_per_image = outputs.logits_per_image  # this is the image-text similarity score
probs = logits_per_image.softmax(
    dim=1
)  # we can take the softmax to get the label probabilities
# TODO: make bar chart that shows the probabilities for each image and each state and if it was correct by looking in the solutions list at given index
print(probs)

In [ ]:
import matplotlib.pyplot as plt

# Get predicted state for each image
pred_indices = probs.argmax(dim=1).tolist()
state_labels = ["open", "closed"]  # order must match prompt

correct = [state_labels[pred] == gt for pred, gt in zip(pred_indices, solutions)]

# Plot
plt.figure(figsize=(12, 4))
for i, (prob, gt, is_correct) in enumerate(zip(probs, solutions, correct)):
    pred_label = state_labels[prob.argmax().item()]
    color = "green" if is_correct else "red"
    plt.bar(i, prob.max().item(), color=color)
    plt.text(i, prob.max().item(), f"{pred_label}", ha="center", va="bottom", fontsize=8)
plt.xlabel("Image index")
plt.ylabel("Predicted probability")
plt.title("CLIP State Classification (green=correct, red=incorrect)")
plt.show()

In [ ]:
# Visualize image embeddings with PCA
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np

# Make sure img_embs and solutions are available from previous cells
# solutions should be a list of "open"/"closed" strings

# Convert solutions to colors
colors = ["green" if s == "open" else "red" for s in solutions]

# Reduce to 2D with PCA
pca = PCA(n_components=2)
img_embs_2d = pca.fit_transform(outputs.cpu().numpy())

plt.figure(figsize=(8, 6))
plt.scatter(img_embs_2d[:, 0], img_embs_2d[:, 1], c=colors, alpha=0.7)
plt.title("PCA of Image Embeddings (green=open, red=closed)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()